In [2]:
# ============================================================
# 05_sarima_rolling_eval.ipynb

# SARIMA / Airline / autoARIMA rolling-origin evaluation
#
# Methodology:
#   - Train: 2014-2022, imputed training series
#   - Test: 2023, raw observed truth
#   - Model: one model per country and disease
#   - Fixed specs:
#       SARIMA(1,0,0)x(1,0,0)[52]
#       SARIMA(1,0,0)x(1,1,0)[52]
#       Airline SARIMA(0,1,1)x(0,1,1)[52]
#   - Optional autoARIMA:
#       orders selected once on training
#   - Rolling evaluation:
#       current 2023 week is appended to the state-space filter
#       even if missing, using NaN to advance the filter calendar
#   - Evaluation:
#       explicit target calendar, auditable n_points
# ============================================================

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project imports
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_project_paths
from src.io_data import read_ari_ili
from src.gaps import gap_summary
from src.calendars import make_calendars_from_df, extract_raw_weekly_series
from src.impute import impute_series_weekly
from src.metrics import mae, rmse, mase_scale_seasonal, mase_from_mae

from statsmodels.tsa.statespace.sarimax import SARIMAX

# Optional: autoARIMA
try:
    from pmdarima.arima import auto_arima
    PMDARIMA_OK = True
except ImportError:
    PMDARIMA_OK = False


# ============================================================
# 0. Configuration
# ============================================================

MIN_OBS = 300
MAX_GAP = 8
M_SEASON = 52

HORIZONS = (1, 2, 3, 4)
MAX_H = max(HORIZONS)

SARIMA_SPECS = [
    (
        "SARIMA(1,0,0)x(1,0,0)[52] (initial)",
        (1, 0, 0),
        (1, 0, 0, 52),
        "c"
    ),
    (
        "SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",
        (1, 0, 0),
        (1, 1, 0, 52),
        "n"
    ),
    (
        "Airline SARIMA(0,1,1)x(0,1,1)[52]",
        (0, 1, 1),
        (0, 1, 1, 52),
        "n"
    ),
]


# ============================================================
# 1. Load data
# ============================================================

paths = get_project_paths(PROJECT_ROOT)
paths.results.mkdir(parents=True, exist_ok=True)

ari_df, ili_df = read_ari_ili(paths.data)

ari_df["truth_date"] = pd.to_datetime(ari_df["truth_date"])
ili_df["truth_date"] = pd.to_datetime(ili_df["truth_date"])

ari_df = ari_df.sort_values(["location", "truth_date"]).reset_index(drop=True)
ili_df = ili_df.sort_values(["location", "truth_date"]).reset_index(drop=True)

print("ARI shape:", ari_df.shape)
print("ILI shape:", ili_df.shape)


# ============================================================
# 2. Valid countries
# ============================================================

gap_ari = gap_summary(ari_df, disease="ARI", calendar_mode="data_range")
gap_ili = gap_summary(ili_df, disease="ILI", calendar_mode="data_range")

ok_ari = gap_ari[
    (gap_ari["n_observed"] >= MIN_OBS) &
    (gap_ari["max_gap_length"] <= MAX_GAP)
]["location"].tolist()

ok_ili = gap_ili[
    (gap_ili["n_observed"] >= MIN_OBS) &
    (gap_ili["max_gap_length"] <= MAX_GAP)
]["location"].tolist()

ok_ari = sorted(ok_ari)
ok_ili = sorted(ok_ili)

print("ok_ari:", ok_ari)
print("ok_ili:", ok_ili)


# ============================================================
# 3. Helpers and corrected SARIMA evaluation
# ============================================================

def pick_autoarima_orders_once(y_train: pd.Series, m=52):
    """
    Select ARIMA orders once on training.
    Returns (order, seasonal_order) or None.
    """
    if not PMDARIMA_OK:
        return None

    y = pd.Series(y_train).dropna().astype(float)

    if len(y) < (m + 20):
        return None

    try:
        model = auto_arima(
            y,
            seasonal=True,
            m=m,
            start_p=0,
            start_q=0,
            max_p=2,
            max_q=2,
            start_P=0,
            start_Q=0,
            max_P=1,
            max_Q=1,
            d=None,
            D=None,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            information_criterion="aic"
        )
        return model.order, model.seasonal_order
    except Exception:
        return None


def expected_points_from_truth(y_test: pd.Series, disease: str, location: str):
    rows = []

    for h in HORIZONS:
        n_expected = 0

        for origin in y_test.index:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            if pd.notna(y_test.loc[target]):
                n_expected += 1

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "expected_actual_points": int(n_expected)
        })

    return pd.DataFrame(rows)


def fit_sarimax_once(history_init: pd.Series, order, seasonal_order, trend):
    y = pd.Series(history_init).astype(float).copy()

    model = SARIMAX(
        y,
        order=order,
        seasonal_order=seasonal_order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    res = model.fit(disp=False)
    converged = bool(res.mle_retvals.get("converged", False))

    return res, converged


def rolling_sarima_fast(
    history_init: pd.Series,
    y_test: pd.Series,
    order,
    seasonal_order,
    trend="n",
    model_name="SARIMA",
    disease=None,
    location=None
):
    """
    Correct rolling-origin SARIMA evaluation.

    
      - Fit once on training.
      - At every 2023 origin, append one observation to the state-space filter.
      - If the current observation is missing, append np.nan.
      - Use list-style append [obs_value], not date-indexed Series append.
      - Store all targets inside 2023.
      - Do not filter missing y/pred here.
    """

    rows = []

    try:
        res, converged = fit_sarimax_once(
            history_init=history_init,
            order=order,
            seasonal_order=seasonal_order,
            trend=trend
        )
        fit_ok = True
        fit_error = ""
    except Exception as e:
        res = None
        converged = False
        fit_ok = False
        fit_error = str(e)[:300]

    for origin in y_test.index:

        if res is not None:
            y_obs = y_test.loc[origin]
            obs_value = float(y_obs) if pd.notna(y_obs) else np.nan

            try:
                #  list-style append, not date-indexed Series
                res = res.append([obs_value], refit=False)
            except Exception:
                pass

            try:
                y_hat = res.forecast(steps=MAX_H)
            except Exception:
                y_hat = pd.Series(np.repeat(np.nan, MAX_H))
        else:
            y_hat = pd.Series(np.repeat(np.nan, MAX_H))

        for h in HORIZONS:
            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            actual = y_test.loc[target]

            try:
                pred = y_hat.iloc[h - 1]
            except Exception:
                pred = np.nan

            rows.append({
                "origin": origin,
                "target": target,
                "disease": disease,
                "location": location,
                "h": int(h),
                "y": float(actual) if pd.notna(actual) else np.nan,
                "pred": float(pred) if pd.notna(pred) else np.nan,
                "model": model_name,
                "order": str(order),
                "seasonal_order": str(seasonal_order),
                "trend": trend,
                "fit_ok": int(fit_ok),
                "converged": int(converged),
                "fit_error": fit_error
            })

    return pd.DataFrame(rows)


def summarize_eval(
    df_eval: pd.DataFrame,
    disease: str,
    location: str,
    model_name: str,
    scale: float
):
    rows = []

    for h in HORIZONS:
        sub = df_eval[df_eval["h"] == h].copy()

        sub = sub[sub["y"].notna()].copy()
        sub = sub[sub["pred"].notna()].copy()

        if len(sub) == 0:
            continue

        m_mae = mae(sub["y"].values, sub["pred"].values)
        m_rmse = rmse(sub["y"].values, sub["pred"].values)
        m_mase = mase_from_mae(m_mae, scale)

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "model": model_name,
            "MAE": float(m_mae),
            "RMSE": float(m_rmse),
            "MASE": float(m_mase),
            "n": int(len(sub))
        })

    return pd.DataFrame(rows)


def run_country_sarima_suite(
    df: pd.DataFrame,
    location: str,
    disease: str,
    m_season=52
):
    cals = make_calendars_from_df(df)

    train_imputed = impute_series_weekly(
        df[df["location"] == location][["truth_date", "value"]],
        calendar=cals.train_weeks,
        trim_to_first_obs=True,
        max_gap_knn=8
    )

    y_test = extract_raw_weekly_series(df, location, cals.test_weeks)

    scale = mase_scale_seasonal(train_imputed, m=m_season)

    expected_df = expected_points_from_truth(
        y_test=y_test,
        disease=disease,
        location=location
    )

    country_h_parts = []
    long_parts = []
    status_rows = []

    # Fixed SARIMA specifications
    for name, order, sorder, trend in SARIMA_SPECS:

        df_eval = rolling_sarima_fast(
            history_init=train_imputed,
            y_test=y_test,
            order=order,
            seasonal_order=sorder,
            trend=trend,
            model_name=name,
            disease=disease,
            location=location
        )

        long_parts.append(df_eval)

        status_rows.append({
            "disease": disease,
            "location": location,
            "model": name,
            "selected_order": str(order),
            "selected_seasonal_order": str(sorder),
            "trend": trend,
            "fit_ok": int(df_eval["fit_ok"].max()) if len(df_eval) else 0,
            "converged": int(df_eval["converged"].max()) if len(df_eval) else 0,
            "fit_error": df_eval["fit_error"].iloc[0] if len(df_eval) else "no rows"
        })

        country_h_parts.append(
            summarize_eval(
                df_eval=df_eval,
                disease=disease,
                location=location,
                model_name=name,
                scale=scale
            )
        )

    # autoARIMA once per country
    auto = pick_autoarima_orders_once(train_imputed, m=m_season)

    if auto is not None:
        order, sorder = auto

        
        # For comparison, the model name is generic.
        # Orders are stored separately in long/status files.
        name = "autoARIMA"

        df_eval = rolling_sarima_fast(
            history_init=train_imputed,
            y_test=y_test,
            order=order,
            seasonal_order=sorder,
            trend="n",
            model_name=name,
            disease=disease,
            location=location
        )

        long_parts.append(df_eval)

        status_rows.append({
            "disease": disease,
            "location": location,
            "model": name,
            "selected_order": str(order),
            "selected_seasonal_order": str(sorder),
            "trend": "n",
            "fit_ok": int(df_eval["fit_ok"].max()) if len(df_eval) else 0,
            "converged": int(df_eval["converged"].max()) if len(df_eval) else 0,
            "fit_error": df_eval["fit_error"].iloc[0] if len(df_eval) else "no rows"
        })

        country_h_parts.append(
            summarize_eval(
                df_eval=df_eval,
                disease=disease,
                location=location,
                model_name=name,
                scale=scale
            )
        )

    country_h = (
        pd.concat(country_h_parts, ignore_index=True)
        if len(country_h_parts)
        else pd.DataFrame(columns=["disease", "location", "h", "model", "MAE", "RMSE", "MASE", "n"])
    )

    long_df = (
        pd.concat(long_parts, ignore_index=True)
        if len(long_parts)
        else pd.DataFrame()
    )

    status_df = pd.DataFrame(status_rows)

    return country_h, long_df, expected_df, status_df


def run_sarima_for_disease(df, disease, ok_locations):
    all_country_h = []
    all_long = []
    all_expected = []
    all_status = []

    for loc in ok_locations:
        print(f"\n==================== {disease} {loc} ====================")

        try:
            country_h, long_df, expected_df, status_df = run_country_sarima_suite(
                df=df,
                location=loc,
                disease=disease,
                m_season=M_SEASON
            )

            all_country_h.append(country_h)
            all_long.append(long_df)
            all_expected.append(expected_df)
            all_status.append(status_df)

            print("country_h rows:", len(country_h))

        except Exception as e:
            print(f"Skipping {disease} {loc}: {e}")

    country_h_all = (
        pd.concat(all_country_h, ignore_index=True)
        if all_country_h
        else pd.DataFrame()
    )

    long_all = (
        pd.concat(all_long, ignore_index=True)
        if all_long
        else pd.DataFrame()
    )

    expected_all = (
        pd.concat(all_expected, ignore_index=True)
        if all_expected
        else pd.DataFrame()
    )

    status_all = (
        pd.concat(all_status, ignore_index=True)
        if all_status
        else pd.DataFrame()
    )

    if not country_h_all.empty:
        scored = (
            country_h_all
            .groupby(["disease", "location", "h", "model"], as_index=False)
            .agg(scored_points=("n", "sum"))
        )

        models = country_h_all[["disease", "location", "model"]].drop_duplicates()

        expected_by_model = expected_all.merge(
            models,
            on=["disease", "location"],
            how="inner"
        )

        n_audit = expected_by_model.merge(
            scored,
            on=["disease", "location", "h", "model"],
            how="left"
        )

        n_audit["scored_points"] = n_audit["scored_points"].fillna(0).astype(int)
        n_audit["missing_due_to_model_or_pred_nan"] = (
            n_audit["expected_actual_points"] - n_audit["scored_points"]
        )

    else:
        n_audit = pd.DataFrame()

    return country_h_all, long_all, expected_all, n_audit, status_all


# ============================================================
# 4. Run ARI and ILI
# ============================================================

sarima_ari, sarima_ari_long, sarima_ari_expected, sarima_ari_n_audit, sarima_ari_status = run_sarima_for_disease(
    df=ari_df,
    disease="ARI",
    ok_locations=ok_ari
)

sarima_ili, sarima_ili_long, sarima_ili_expected, sarima_ili_n_audit, sarima_ili_status = run_sarima_for_disease(
    df=ili_df,
    disease="ILI",
    ok_locations=ok_ili
)

print("\nARI SARIMA country_h")
display(sarima_ari.head())
print("sarima_ari shape:", sarima_ari.shape)

print("\nILI SARIMA country_h")
display(sarima_ili.head())
print("sarima_ili shape:", sarima_ili.shape)


# ============================================================
# 5. Audits
# ============================================================

print("\nARI SARIMA n audit aggregated by model")
if not sarima_ari_n_audit.empty:
    display(
        sarima_ari_n_audit
        .groupby(["model", "h"], as_index=False)
        .agg(
            expected_actual_points=("expected_actual_points", "sum"),
            scored_points=("scored_points", "sum"),
            missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
        )
        .sort_values(["model", "h"])
    )

print("\nILI SARIMA n audit aggregated by model")
if not sarima_ili_n_audit.empty:
    display(
        sarima_ili_n_audit
        .groupby(["model", "h"], as_index=False)
        .agg(
            expected_actual_points=("expected_actual_points", "sum"),
            scored_points=("scored_points", "sum"),
            missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
        )
        .sort_values(["model", "h"])
    )

print("\nARI SARIMA model status")
display(sarima_ari_status)

print("\nILI SARIMA model status")
display(sarima_ili_status)


# ============================================================
# 6. Coverage-aware summaries
# ============================================================

def coverage_aware_summary(df_res: pd.DataFrame):
    if df_res.empty:
        return df_res

    out = (
        df_res
        .groupby(["h", "model"], as_index=False)
        .agg(
            n_countries=("location", "nunique"),
            n_points=("n", "sum"),
            mean_MAE=("MAE", "mean"),
            mean_RMSE=("RMSE", "mean"),
            mean_MASE=("MASE", "mean")
        )
        .sort_values(["h", "mean_MASE", "mean_MAE"])
        .reset_index(drop=True)
    )

    return out


sarima_ari_summary = coverage_aware_summary(sarima_ari)
sarima_ili_summary = coverage_aware_summary(sarima_ili)

print("\nARI SARIMA coverage-aware summary")
display(sarima_ari_summary)

print("\nILI SARIMA coverage-aware summary")
display(sarima_ili_summary)


# ============================================================
# 7. Save outputs
# ============================================================

sarima_ari.to_csv(paths.results / "sarima_ari_suite_rolling_2023.csv", index=False)
sarima_ili.to_csv(paths.results / "sarima_ili_suite_rolling_2023.csv", index=False)

sarima_ari_long.to_csv(paths.results / "sarima_ari_suite_rolling_2023_long.csv", index=False)
sarima_ili_long.to_csv(paths.results / "sarima_ili_suite_rolling_2023_long.csv", index=False)

sarima_ari_expected.to_csv(paths.results / "sarima_ari_expected_points_2023.csv", index=False)
sarima_ili_expected.to_csv(paths.results / "sarima_ili_expected_points_2023.csv", index=False)

sarima_ari_n_audit.to_csv(paths.results / "sarima_ari_n_audit_2023.csv", index=False)
sarima_ili_n_audit.to_csv(paths.results / "sarima_ili_n_audit_2023.csv", index=False)

sarima_ari_status.to_csv(paths.results / "sarima_ari_model_status_2023.csv", index=False)
sarima_ili_status.to_csv(paths.results / "sarima_ili_model_status_2023.csv", index=False)

sarima_ari_summary.to_csv(paths.results / "sarima_ari_horizon_summary_2023.csv", index=False)
sarima_ili_summary.to_csv(paths.results / "sarima_ili_horizon_summary_2023.csv", index=False)

print("\nSaved SARIMA outputs to:", paths.results)

ARI shape: (6685, 4)
ILI shape: (10646, 4)
ok_ari: ['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI']
ok_ili: ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI']

==================== ARI BE ====================
country_h rows: 16

==================== ARI BG ====================
country_h rows: 16

==================== ARI CZ ====================
country_h rows: 16

==================== ARI DE ====================
country_h rows: 16

==================== ARI EE ====================
country_h rows: 16

==================== ARI LT ====================
country_h rows: 16

==================== ARI RO ====================
country_h rows: 16

==================== ARI SI ====================
country_h rows: 16

==================== ILI BE ====================
country_h rows: 16

==================== ILI CZ ====================
country_h rows: 16

==================== ILI EE ====================
country_h rows: 16

==================== ILI GR ====================
country_h rows: 16

==

,disease,location,h,model,MAE,RMSE,MASE,n
0,ARI,BE,1,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",157.786007,208.376026,0.531619,52
1,ARI,BE,2,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",180.971706,230.524258,0.609738,51
2,ARI,BE,3,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",184.526463,254.240743,0.621714,50
3,ARI,BE,4,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",198.952185,269.481572,0.670318,49
4,ARI,BE,1,"SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",190.425361,246.898840,0.641589,52


sarima_ari shape: (128, 8)

ILI SARIMA country_h


,disease,location,h,model,MAE,RMSE,MASE,n
0,ILI,BE,1,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",48.188543,76.714024,0.453388,52
1,ILI,BE,2,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",66.386750,100.714045,0.624608,51
2,ILI,BE,3,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",78.179826,110.359200,0.735565,50
3,ILI,BE,4,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",78.839456,107.451011,0.741771,49
4,ILI,BE,1,"SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",57.963383,86.771648,0.545356,52


sarima_ili shape: (160, 8)

ARI SARIMA n audit aggregated by model


...


ARI SARIMA coverage-aware summary


,h,model,n_countries,n_points,mean_MAE,mean_RMSE,mean_MASE
0,1,autoARIMA,8,409,115.040384,160.781004,0.562527
1,1,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",8,409,116.176472,161.267992,0.568493
2,1,"Airline SARIMA(0,1,1)x(0,1,1)[52]",8,409,123.359584,175.015918,0.590486
3,1,"SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",8,409,123.345949,167.771970,0.596774
4,2,autoARIMA,8,401,152.667170,210.159315,0.743959
5,2,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",8,401,157.152585,214.829296,0.762730
6,2,"SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",8,401,157.111562,211.870289,0.770043
7,2,"Airline SARIMA(0,1,1)x(0,1,1)[52]",8,401,168.524954,239.053920,0.800683
8,3,autoARIMA,8,393,178.268749,242.170522,0.863737
9,3,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",8,393,178.359151,244.706539,0.864808



ILI SARIMA coverage-aware summary


,h,model,n_countries,n_points,mean_MAE,mean_RMSE,mean_MASE
0,1,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",10,513,29.519347,49.479703,0.569590
1,1,autoARIMA,10,513,30.675418,51.524583,0.598145
2,1,"SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",10,513,31.781105,49.822270,0.646910
3,1,"Airline SARIMA(0,1,1)x(0,1,1)[52]",10,513,32.932360,54.437552,0.670962
4,2,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",10,503,39.703946,61.478136,0.754154
5,2,autoARIMA,10,503,40.478850,63.784550,0.757562
6,2,"SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)",10,503,41.145040,62.182111,0.809141
7,2,"Airline SARIMA(0,1,1)x(0,1,1)[52]",10,503,44.687473,74.516089,0.922180
8,3,autoARIMA,10,493,49.629488,77.944801,0.888936
9,3,"SARIMA(1,0,0)x(1,0,0)[52] (initial)",10,493,48.239247,73.352338,0.906188



Saved SARIMA outputs to: C:\Users\aolas\UNI PYTHON\TFG\results
